# 08 - Concert Ranking Engine

## Goal

Build the first explainable GigRoute recommendation model by combining music preferences with geographic distance.

## Tasks

- Load concert and artist enrichment data
- Define test user preferences
- Filter concerts by date and travel radius
- Calculate artist preference score
- Calculate genre relevance score
- Calculate distance score
- Combine signals into a final ranking score
- Validate and explain recommendation ordering

In [5]:
import os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text

In [9]:
project_path = Path("..")

load_dotenv(project_path / ".env", override=True)

db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")
db_name = os.getenv("POSTGRES_DB")
db_port = os.getenv("POSTGRES_PORT")

## Database Connection

The PostgreSQL/PostGIS database is used to filter concerts geographically before recommendation scoring.

In [11]:
database_url = URL.create(
    drivername= "postgresql+psycopg2",
    username = db_user,
    password = db_password,
    host = "localhost",
    port = int(db_port),
    database=db_name
)

engine = create_engine(database_url)

In [17]:

# connection test

with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM events")
    )
    event_count = result.scalar()

In [18]:
print("Events in database:", event_count)

Events in database: 1172


## Load Artist Enrichment Data

The final artist identity mapping and MusicBrainz genre data are loaded for recommendation scoring.

In [23]:
musicbrainz_path = (project_path / "data" / "processed" / "musicbrainz")

artist_mapping = pd.read_csv(
    musicbrainz_path / "artist_mapping.csv"
)

artist_genres = pd.read_csv(
    musicbrainz_path / "artist_genres.csv"
)

In [24]:
print("Artist mappings:", artist_mapping.shape)
print("Artist genres:", artist_genres.shape)

Artist mappings: (472, 19)
Artist genres: (785, 3)


In [25]:
artist_mapping["match_status"].value_counts()

match_status
auto_match         293
ambiguous           82
no_candidate        72
unresolved          13
secondary_match     12
Name: count, dtype: int64

## Test User Preferences

A reproducible test profile is used to validate the ranking logic before it is moved into the application.

In [26]:
user_latitude = 52.5200
user_longitude = 13.4050

radius_km = 300

start_date = "2026-08-21"
end_date = "2027-02-28"

In [27]:
preferred_artists = (
    artist_mapping.loc[
        artist_mapping["mbid"].notna(),
        "ticketmaster_artist_name"
    ]
    .drop_duplicates()
    .head(3)
    .tolist()
)

preferred_artists

['Heavysaurus', 'Rawayana', 'Dance Gavin Dance']

In [28]:
preferred_genres = (
    artist_genres["genre"]
    .value_counts()
    .head(3)
    .index
    .tolist()
)

preferred_genres

['pop', 'rock', 'hip hop']

## Geographic Candidate Selection

PostGIS filters events by travel radius and calculates the distance from the test user's location.

In [29]:
events_query = text("""
    SELECT
        event_id,
        event_name,
        artist_name,
        event_date,
        event_time,
        venue_name,
        city,
        country,
        latitude,
        longitude,
        ST_Distance(
            location,
            ST_SetSRID(
                ST_MakePoint(
                    :user_longitude,
                    :user_latitude
                ),
                4326
            )::geography
        ) / 1000 AS distance_km
    FROM events
    WHERE event_date BETWEEN :start_date AND :end_date
      AND ST_DWithin(
            location,
            ST_SetSRID(
                ST_MakePoint(
                    :user_longitude,
                    :user_latitude
                ),
                4326
            )::geography,
            :radius_meters
      )
    ORDER BY event_date;
""")

In [30]:
events_df = pd.read_sql(
    events_query,
    engine,
    params={
        "user_longitude": user_longitude,
        "user_latitude": user_latitude,
        "radius_meters": radius_km * 1000,
        "start_date": start_date,
        "end_date": end_date
    }
)

In [33]:
events_df.head()
events_df.shape

events_df["distance_km"].describe()

count    523.000000
mean     109.427239
std      118.528368
min        1.072681
25%        3.088968
50%        6.482444
75%      257.635170
max      295.985652
Name: distance_km, dtype: float64

In [34]:
print("Nearby events:", len(events_df))

print(
    "Maximum distance:",
    events_df["distance_km"].max()
)

Nearby events: 523
Maximum distance: 295.98565240878


In [35]:
print(
    "All events within radius:",
    events_df["distance_km"]
    .le(radius_km)
    .all()
)

All events within radius: True
